In [1]:
!nvidia-smi


Sat Apr 18 10:20:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%pip install transformers==4.40.0 IndicTransToolkit datasets
%pip install torch torchvision torchaudio --index-url https://pytorch.org


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 94.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 546.3/546.3 kB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 123.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 106.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 15.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.10.1
    Uninstalling huggingface_hub-1.10.1:
      Successfully uninstalled huggingface_hub-1.10

Looking in indexes: https://pytorch.org


In [3]:
import os
import json
import torch
from datasets import load_dataset
from pathlib import Path
from IndicTransToolkit import IndicProcessor
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# ────────────────────────────────────────
# CONFI
# ─────────────────────────────────────────
MODEL_NAME   = "ai4bharat/indictrans2-indic-en-1B"
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR   = Path("data/translated")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE   = 8        # sentences per GPU batch — tune if you get OOM
MAX_SENTENCES = 2   # set to e.g. 50 to do a quick test run first

print(f"Using device: {DEVICE}")

# ─────────────────────────────────────────
# LOAD MODEL & TOOLKIT
# ─────────────────────────────────────────
print("Loading IndicTrans2 model and toolkit...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model_it2 = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, trust_remote_code=True).to(DEVICE)
ip        = IndicProcessor(inference=True)
print("Model loaded successfully.")

# ─────────────────────────────────────────
# LOAD DATASET
# ─────────────────────────────────────────
print("Loading dataset...")
ds = load_dataset("Exploration-Lab/IL-TUR", "bail")

# We use train_all for training, test_all for evaluation
splits_to_translate = {
    "train": ds["train_all"],
    "test":  ds["test_all"],
    "dev":   ds["dev_all"],
}

# ─────────────────────────────────────────
# HELPER: Translate a flat list of sentences in batches
# ─────────────────────────────────────────
def translate_sentences(sentences: list[str], src_lang="hin_Deva") -> list[str]:
    """
    Translate a list of Hindi sentences to English using IndicTrans2 + Toolkit.
    """
    translated = []
    for i in range(0, len(sentences), BATCH_SIZE):
        # Prepare the batch and handle empty strings
        batch = [s.strip() if s.strip() else "." for s in sentences[i : i + BATCH_SIZE]]
        
        try:
            # 1. Preprocess using the toolkit
            inputs = ip.preprocess_batch(batch, src_lang=src_lang, tgt_lang="eng_Latn")
            
            # 2. Tokenize for the model
            tokenized = tokenizer(inputs, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
            
            # 3. Generate translation tokens
            with torch.no_grad():
                output_tokens = model_it2.generate(**tokenized, num_beams=4, max_length=256)
            
            # 4. Decode and Post-process
            result = tokenizer.batch_decode(output_tokens, skip_special_tokens=True)
            result = ip.postprocess_batch(result, lang="eng_Latn")
            
            translated.extend(result)
            
        except Exception as e:
            print(f"  ⚠ Translation error on batch {i//BATCH_SIZE}: {e}")
            translated.extend(["[TRANSLATION ERROR]"] * len(batch))
            
    return translated


# ─────────────────────────────────────────
# HELPER: Detect language (simple heuristic)
# ─────────────────────────────────────────
def detect_src_lang(text: str) -> str:
    """
    Returns IndicTrans2 language code.
    Default to Hindi (Devanagari).
    """
    devanagari_chars = sum(1 for c in text if '\u0900' <= c <= '\u097F')
    if devanagari_chars > 5:
        return "hin_Deva"
    return "hin_Deva"


# ─────────────────────────────────────────
# MAIN TRANSLATION LOOP
# ─────────────────────────────────────────
for split_name, split_data in splits_to_translate.items():
    output_path = OUTPUT_DIR / f"{split_name}.json"

    # Resume if already done
    if output_path.exists():
        print(f"✓ {split_name} already translated, skipping.")
        continue

    print(f"\n{'='*50}")
    print(f"Translating split: {split_name} ({len(split_data)} cases)")
    print(f"{'='*50}")

    translated_cases = []

    for idx, case in enumerate(split_data):
        if MAX_SENTENCES and idx >= MAX_SENTENCES:
            break

        case_id  = case["id"]
        district = case["district"]
        label    = case["label"]
        facts    = case["text"]["facts-and-arguments"]
        opinion  = case["text"]["judge-opinion"]

        print(f"  [{idx+1}/{len(split_data)}] Case {case_id} | district={district} | label={label}")

        # Detect language from first non-empty sentence
        sample_text = next((s for s in facts + opinion if s.strip()), "")
        src_lang = detect_src_lang(sample_text)

        # Translate both sections
        translated_facts   = translate_sentences(facts,   src_lang=src_lang)
        translated_opinion = translate_sentences(opinion, src_lang=src_lang)

        translated_cases.append({
            "id":       case_id,
            "district": district,
            "label":    label,
            "label_str": "DENIED" if label == 0 else "GRANTED",
            "text": {
                "facts-and-arguments": translated_facts,
                "judge-opinion":       translated_opinion,
            },
            "original_text": {
                "facts-and-arguments": facts,
                "judge-opinion":       opinion,
            }
        })

        # Save incrementally every 50 cases
        if (idx + 1) % 50 == 0:
            with open(output_path, "w", encoding="utf-8") as f:
                json.dump(translated_cases, f, ensure_ascii=False, indent=2)
            print(f"    💾 Checkpoint saved ({idx+1} cases)")

    # Final save for the split
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(translated_cases, f, ensure_ascii=False, indent=2)

    print(f"\n✅ Done: {split_name} → {output_path} ({len(translated_cases)} cases)")

print("\n🎉 All splits translated successfully.")


Using device: cuda
Loading IndicTrans2 model and toolkit...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/ai4bharat/indictrans2-indic-en-1B.
401 Client Error. (Request ID: Root=1-69e35c09-70b635f915a7461c67eeec26;d055698e-4e5b-474f-9198-64f41ff28e0d)

Cannot access gated repo for url https://huggingface.co/ai4bharat/indictrans2-indic-en-1B/resolve/main/config.json.
Access to model ai4bharat/indictrans2-indic-en-1B is restricted. You must have access to it and be authenticated to access it. Please log in.

In [4]:
from huggingface_hub import login
login(new_session=False)


In [5]:
%pip install huggingface_hub   

In [6]:
import os
import json
import torch
from datasets import load_dataset
from pathlib import Path
from IndicTransToolkit import IndicProcessor
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# ────────────────────────────────────────
# CONFI
# ─────────────────────────────────────────
MODEL_NAME   = "ai4bharat/indictrans2-indic-en-1B"
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR   = Path("data/translated")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE   = 8        # sentences per GPU batch — tune if you get OOM
MAX_SENTENCES = 2   # set to e.g. 50 to do a quick test run first

print(f"Using device: {DEVICE}")

# ─────────────────────────────────────────
# LOAD MODEL & TOOLKIT
# ─────────────────────────────────────────
print("Loading IndicTrans2 model and toolkit...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model_it2 = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, trust_remote_code=True).to(DEVICE)
ip        = IndicProcessor(inference=True)
print("Model loaded successfully.")

# ─────────────────────────────────────────
# LOAD DATASET
# ─────────────────────────────────────────
print("Loading dataset...")
ds = load_dataset("Exploration-Lab/IL-TUR", "bail")

# We use train_all for training, test_all for evaluation
splits_to_translate = {
    "train": ds["train_all"],
    "test":  ds["test_all"],
    "dev":   ds["dev_all"],
}

# ─────────────────────────────────────────
# HELPER: Translate a flat list of sentences in batches
# ─────────────────────────────────────────
def translate_sentences(sentences: list[str], src_lang="hin_Deva") -> list[str]:
    """
    Translate a list of Hindi sentences to English using IndicTrans2 + Toolkit.
    """
    translated = []
    for i in range(0, len(sentences), BATCH_SIZE):
        # Prepare the batch and handle empty strings
        batch = [s.strip() if s.strip() else "." for s in sentences[i : i + BATCH_SIZE]]
        
        try:
            # 1. Preprocess using the toolkit
            inputs = ip.preprocess_batch(batch, src_lang=src_lang, tgt_lang="eng_Latn")
            
            # 2. Tokenize for the model
            tokenized = tokenizer(inputs, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
            
            # 3. Generate translation tokens
            with torch.no_grad():
                output_tokens = model_it2.generate(**tokenized, num_beams=4, max_length=256)
            
            # 4. Decode and Post-process
            result = tokenizer.batch_decode(output_tokens, skip_special_tokens=True)
            result = ip.postprocess_batch(result, lang="eng_Latn")
            
            translated.extend(result)
            
        except Exception as e:
            print(f"  ⚠ Translation error on batch {i//BATCH_SIZE}: {e}")
            translated.extend(["[TRANSLATION ERROR]"] * len(batch))
            
    return translated


# ─────────────────────────────────────────
# HELPER: Detect language (simple heuristic)
# ─────────────────────────────────────────
def detect_src_lang(text: str) -> str:
    """
    Returns IndicTrans2 language code.
    Default to Hindi (Devanagari).
    """
    devanagari_chars = sum(1 for c in text if '\u0900' <= c <= '\u097F')
    if devanagari_chars > 5:
        return "hin_Deva"
    return "hin_Deva"


# ─────────────────────────────────────────
# MAIN TRANSLATION LOOP
# ─────────────────────────────────────────
for split_name, split_data in splits_to_translate.items():
    output_path = OUTPUT_DIR / f"{split_name}.json"

    # Resume if already done
    if output_path.exists():
        print(f"✓ {split_name} already translated, skipping.")
        continue

    print(f"\n{'='*50}")
    print(f"Translating split: {split_name} ({len(split_data)} cases)")
    print(f"{'='*50}")

    translated_cases = []

    for idx, case in enumerate(split_data):
        if MAX_SENTENCES and idx >= MAX_SENTENCES:
            break

        case_id  = case["id"]
        district = case["district"]
        label    = case["label"]
        facts    = case["text"]["facts-and-arguments"]
        opinion  = case["text"]["judge-opinion"]

        print(f"  [{idx+1}/{len(split_data)}] Case {case_id} | district={district} | label={label}")

        # Detect language from first non-empty sentence
        sample_text = next((s for s in facts + opinion if s.strip()), "")
        src_lang = detect_src_lang(sample_text)

        # Translate both sections
        translated_facts   = translate_sentences(facts,   src_lang=src_lang)
        translated_opinion = translate_sentences(opinion, src_lang=src_lang)

        translated_cases.append({
            "id":       case_id,
            "district": district,
            "label":    label,
            "label_str": "DENIED" if label == 0 else "GRANTED",
            "text": {
                "facts-and-arguments": translated_facts,
                "judge-opinion":       translated_opinion,
            },
            "original_text": {
                "facts-and-arguments": facts,
                "judge-opinion":       opinion,
            }
        })

        # Save incrementally every 50 cases
        if (idx + 1) % 50 == 0:
            with open(output_path, "w", encoding="utf-8") as f:
                json.dump(translated_cases, f, ensure_ascii=False, indent=2)
            print(f"    💾 Checkpoint saved ({idx+1} cases)")

    # Final save for the split
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(translated_cases, f, ensure_ascii=False, indent=2)

    print(f"\n✅ Done: {split_name} → {output_path} ({len(translated_cases)} cases)")

print("\n🎉 All splits translated successfully.")


Using device: cuda
Loading IndicTrans2 model and toolkit...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-1B:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

Generating test_all split:   0%|          | 0/35400 [00:00<?, ? examples/s]

Generating train_specific split:   0%|          | 0/124341 [00:00<?, ? examples/s]

Generating dev_specific split:   0%|          | 0/15929 [00:00<?, ? examples/s]

Generating test_specific split:   0%|          | 0/36579 [00:00<?, ? examples/s]


Translating split: train (123742 cases)
  [1/123742] Case Bail Application_2008_202101-04-2021407 | district=agra | label=0
  [2/123742] Case Bail Application_1081_202111-02-20212541 | district=agra | label=1

✅ Done: train → data/translated/train.json (2 cases)

Translating split: test (35400 cases)
  [1/35400] Case Bail Application_473_202129-01-20213091 | district=agra | label=1
  [2/35400] Case Bail Application_2568_202030-09-20202372 | district=agra | label=1

✅ Done: test → data/translated/test.json (2 cases)

Translating split: dev (17707 cases)
  [1/17707] Case Bail Application_7218_201918-12-2019214 | district=agra | label=1
  [2/17707] Case Bail Application_845_202115-03-20211097 | district=agra | label=0

✅ Done: dev → data/translated/dev.json (2 cases)

🎉 All splits translated successfully.


In [7]:
from google.colab import files
import os

# This lists all files in your translated folder to make sure they exist
path = 'data/translated'
print(f"Files found: {os.listdir(path)}")

# This will trigger a pop-up in your browser to save the file to your laptop
files.download('data/translated/train.json')


Files found: ['test.json', 'train.json', 'dev.json']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
import pandas as pd
import json

# 1. Load the sanity test JSON
with open('data/translated/train.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# 2. Flatten (joining the list of sentences into one block of text)
flattened_data = []
for case in data:
    flattened_data.append({
        "id": case["id"],
        "label": case["label"],
        "translated_facts": " ".join(case["text"]["facts-and-arguments"]),
        "translated_opinion": " ".join(case["text"]["judge-opinion"])
    })

# 3. Save as your specific filename
df = pd.DataFrame(flattened_data)
df.to_csv('train_translated_two.csv', index=False)

# 4. Download
from google.colab import files
files.download('train_translated_two.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
from IPython.display import FileLink
# This creates a blue link in your output window. Just click it!
FileLink('train_translated_two.csv')


/content/train_translated_two.csv

In [ ]:
from google.colab import files
import os

# This lists all files in your translated folder to make sure they exist
path = 'data/translated'
print(f"Files found: {os.listdir(path)}")

# This will trigger a pop-up in your browser to save the file to your laptop
files.download('data/translated/train.json')


FileNotFoundError: [Errno 2] No such file or directory: 'backend/scripts/data/translated'